# 04 XGBoost Forecasting

Train one XGBoostRegressor per forecast horizon using chronological train/test splits.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from xgboost import XGBRegressor

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from src.evaluate import evaluate_model
from src.features import get_feature_columns
from src.train_utils import save_feature_columns, save_model, time_series_train_test_split

PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
model_df = pd.read_csv(PROCESSED_DIR / "ohio_model_ready.csv", parse_dates=["timestamp"])
train_df, test_df = time_series_train_test_split(model_df, test_size=0.2)
feature_columns = get_feature_columns(model_df)
target_columns = {
    "30min": "target_30min",
    "60min": "target_60min",
    "120min": "target_120min",
}
save_feature_columns(feature_columns, MODELS_DIR / "feature_columns.json")
feature_columns

In [ ]:
results = []
trained_models = {}

for horizon, target_column in target_columns.items():
    model = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(train_df[feature_columns], train_df[target_column])
    trained_models[horizon] = model
    save_model(model, MODELS_DIR / f"xgboost_{horizon}.pkl")
    results.append(evaluate_model(model, test_df, feature_columns, target_column, horizon=horizon))

pd.DataFrame(results)